Reference:

    - Perceiver IO: https://arxiv.org/abs/2107.14795

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl
from torch.utils.data import DataLoader, TensorDataset
from typing import List, Tuple

# --- SECTION 1: BUILDING BLOCK MODULES ---
# These are the reusable components of our architecture.

class PerceiverStackEncoder(nn.Module):
    """
    The encoder for a single stack of input items (e.g., all 'credit/duration' transactions).
    It uses cross-attention to distill a variable-length input stack into a fixed-size
    latent representation, which is then processed by a standard Transformer.
    """
    def __init__(self, input_dim: int, latent_dim: int, num_latents: int, depth: int):
        super().__init__()
        # The learnable latent array that acts as the bottleneck. It's the "interviewer".
        self.latents = nn.Parameter(torch.randn(num_latents, latent_dim))

        # First layer: Cross-attention where the latents attend to the input data.
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=latent_dim, num_heads=4, batch_first=True
        )
        # A linear layer to project the input data to the same dimension as the latents.
        self.cross_proj = nn.Linear(input_dim, latent_dim)

        # The main processing block: a standard Transformer encoder that operates
        # only in the latent space. Its computational cost is independent of input size.
        self.blocks = nn.ModuleList(
            [
                nn.TransformerEncoderLayer(
                    d_model=latent_dim,
                    nhead=4,
                    dim_feedforward=latent_dim * 4,
                    batch_first=True,
                    activation=F.gelu # GELU is common in modern transformers
                )
                for _ in range(depth)
            ]
        )

    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Input tensor of shape [B, N, D_in] (Batch, Num Items, Input Dim)
            mask: Boolean tensor of shape [B, N] where True indicates a valid (non-padded) item.

        Returns:
            A tensor of shape [B, latent_dim] summarizing the entire stack.
        """
        B, N, _ = x.shape

        if x.dim() == 2:
            x = x.unsqueeze(0)  # Ensure batch dim exists
        if mask.dim() == 1:
            mask = mask.unsqueeze(0)

        # Create a batch of the learnable latents for processing.
        latents = self.latents.unsqueeze(0).repeat(B, 1, 1)  # Shape: [B, num_latents, latent_dim]

        # Project the input data to the latent dimension.
        x_proj = self.cross_proj(x)  # Shape: [B, N, latent_dim]

        # The key Perceiver step: cross-attention.
        # Queries come from our learnable latents.
        # Keys/Values come from the projected input data.
        # The key_padding_mask ensures we don't attend to padded parts of the input.
        latents, _ = self.cross_attn(
            latents, x_proj, x_proj, key_padding_mask=~mask
        )

        # Process the distilled information through the deep latent transformer.
        for block in self.blocks:
            latents = block(latents)

        # Pool the final latent vectors into a single summary vector for this stack.
        return latents.mean(dim=1)


class PerceiverDecoder(nn.Module):
    """
    The shared decoder module.
    It takes a shared latent vector (containing info from all 6 stacks) and a specific
    output query to reconstruct a single target stack.
    """
    def __init__(self, shared_dim: int, query_dim: int, output_dim: int, depth: int):
        super().__init__()
        self.output_dim = output_dim

        # Projects the shared latent vector to the decoder's internal dimension.
        self.latent_proj = nn.Linear(shared_dim, query_dim)

        # Cross-attention where the output query "interrogates" the processed latent vector.
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=query_dim, num_heads=4, batch_first=True
        )
        
        # A standard Transformer encoder for deep processing.
        self.blocks = nn.ModuleList(
            [
                nn.TransformerEncoderLayer(
                    d_model=query_dim,
                    nhead=4,
                    dim_feedforward=query_dim * 4,
                    batch_first=True,
                    activation=F.gelu
                )
                for _ in range(depth)
            ]
        )
        # Final linear layer to project the output to the desired reconstruction dimension.
        self.output_head = nn.Linear(query_dim, output_dim)

    def forward(self, shared_latent: torch.Tensor, output_query: torch.Tensor) -> torch.Tensor:
        """
        Args:
            shared_latent: The compressed vector from the encoders. Shape [B, shared_dim].
            output_query: The query specifying which stack to reconstruct. Shape [B, M, query_dim].
                          M is the max number of items to reconstruct.
        Returns:
            The reconstructed stack. Shape [B, M, output_dim].
        """
        # 1. Project the single latent vector to the right dimension.
        # Input shared_latent shape: [B, shared_dim] (e.g., [4, 512])
        latent_proj = self.latent_proj(shared_latent)
        # Output latent_proj shape: [B, query_dim] (e.g., [4, 192])

        # 2. Prepare the latent vector to be the key and value in cross-attention.
        # It needs to be repeated for every item in the output_query sequence.
        # Add a sequence dimension: [B, 1, query_dim] (e.g., [4, 1, 192])
        # Expand along the sequence dimension to match the output_query's length (M).
        # Target shape: [B, M, query_dim] (e.g., [4, 40, 192])
        latent = latent_proj.unsqueeze(1).expand(-1, output_query.size(1), -1)

        # 3. Cross-attention: The output_query asks questions of the latent vector.
        # Query: output_query ([B, M, query_dim])
        # Key:   latent       ([B, M, query_dim])
        # Value: latent       ([B, M, query_dim])
        x, _ = self.cross_attn(output_query, latent, latent)

        # 4. Deep processing of the resulting information.
        for block in self.blocks:
            x = block(x)

        # 5. Project to the final output dimension (e.g., 256).
        return self.output_head(x)

In [ ]:
class Stage2Autoencoder(pl.LightningModule):
    def __init__(
        self,
        num_categories: int = 6, # IMPORTANT: This *must* be 6
        input_dim: int = 256, # IMPORTANT: This *must* be 256
        encoder_latent_dim: int = 128,
        num_latents: int = 64,
        encoder_depth: int = 4,
        shared_latent_dim: int = 512,
        query_dim: int = 192,
        decoder_depth: int = 4,
        lr: float = 1e-4,
    ):
        super().__init__()
        self.save_hyperparameters()

        self.encoders = nn.ModuleList(
            [
                PerceiverStackEncoder(input_dim, encoder_latent_dim, num_latents, encoder_depth)
                for _ in range(num_categories)
            ]
        )
        self.encoder_to_shared = nn.Linear(num_categories * encoder_latent_dim, shared_latent_dim)

        self.balance_embedding = nn.Embedding(num_embeddings=3, embedding_dim=query_dim)
        self.period_embedding = nn.Embedding(num_embeddings=3, embedding_dim=query_dim)

        self.decoder = PerceiverDecoder(shared_latent_dim, query_dim, input_dim, decoder_depth)
        self.loss_fn = nn.MSELoss()

    def encode(self, stacks: List[torch.Tensor], masks: List[torch.Tensor]) -> torch.Tensor:
        encoder_outputs = [
            self.encoders[i](stacks[i], masks[i]) for i in range(self.hparams.num_categories)
        ]
        concatenated_latents = torch.cat(encoder_outputs, dim=1)
        return self.encoder_to_shared(concatenated_latents)

    def decode(
        self,
        shared_latent: torch.Tensor,
        balance_batch: List[torch.Tensor],
        period_batch: List[torch.Tensor]
    ) -> List[torch.Tensor]:
        stack_queries = []
        for i in range(self.hparams.num_categories):
            balance_embed = self.balance_embedding(balance_batch[i])
            period_embed = self.period_embedding(period_batch[i])
            query = balance_embed + period_embed
            stack_queries.append(query)

        return [self.decoder(shared_latent, query) for query in stack_queries]

    def forward(
        self,
        stacks: List[torch.Tensor],
        masks: List[torch.Tensor],
        balance_batch: List[torch.Tensor],
        period_batch: List[torch.Tensor]
    ) -> Tuple[List[torch.Tensor], torch.Tensor]:
        shared_latent = self.encode(stacks, masks)
        reconstructions = self.decode(shared_latent, balance_batch, period_batch)
        return reconstructions, shared_latent

    def training_step(self, batch, batch_idx):
        stacks, masks, balance_batch, period_batch = batch
        reconstructions, _ = self.forward(stacks, masks, balance_batch, period_batch)

        total_loss = 0
        for i in range(self.hparams.num_categories):
            original = stacks[i][masks[i]]
            recon = reconstructions[i][masks[i]]
            if original.numel() > 0:
                loss = self.loss_fn(recon, original)
                total_loss += loss
                self.log(f"train_loss_stack_{i}", loss, on_step=True, on_epoch=True, prog_bar=False)

        self.log("train_total_loss", total_loss, on_step=True, on_epoch=True, prog_bar=True)
        return total_loss

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=self.hparams.lr)


In [ ]:
import os
from pathlib import Path
from pytorch_lightning.loggers import TensorBoardLogger
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from torch.utils.data import DataLoader
import pytorch_lightning as pl

from config import project_paths, simd_r_drive_server_config
# from models.pytorch.narrative_stack.stage2.model import Stage2Autoencoder
from models.pytorch.narrative_stack.stage2.dataset import (
    Stage2StackDataset,
    stage2_collate_stacks,
)

# === CONFIG ===
OUTPUT_PATH = Path(project_paths.python_data / "stage2_training_output")
os.makedirs(OUTPUT_PATH, exist_ok=True)

EPOCHS = 1000
PATIENCE = 20
BATCH_SIZE = 4  # Safe to increase; collate handles padding
NUM_WORKERS = 2
LOOKUP_BATCH_SIZE = 64
GRAD_CLIP = 0.5

# === MODEL ===
model = Stage2Autoencoder()

# === DATALOADERS ===
train_loader = DataLoader(
    Stage2StackDataset(
        simd_r_drive_server_config=simd_r_drive_server_config,
        shuffle=True,
        lookup_batch_size=LOOKUP_BATCH_SIZE,
    ),
    batch_size=BATCH_SIZE,
    collate_fn=stage2_collate_stacks,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4,
    num_workers=NUM_WORKERS,
)

val_loader = DataLoader(
    Stage2StackDataset(
        simd_r_drive_server_config=simd_r_drive_server_config,
        shuffle=False,
        lookup_batch_size=LOOKUP_BATCH_SIZE,
    ),
    batch_size=BATCH_SIZE,
    collate_fn=stage2_collate_stacks,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4,
    num_workers=NUM_WORKERS,
)

# === CALLBACKS ===
early_stop_callback = EarlyStopping(
    monitor="train_total_loss",
    patience=PATIENCE,
    verbose=True,
    mode="min",
)

model_checkpoint = ModelCheckpoint(
    dirpath=OUTPUT_PATH,
    filename="stage2_checkpoint",
    monitor="train_total_loss",
    mode="min",
    save_top_k=1,
    verbose=True,
)

# === TRAINER ===
trainer = pl.Trainer(
    max_epochs=EPOCHS,
    logger=TensorBoardLogger(OUTPUT_PATH, name="stage2_autoencoder"),
    callbacks=[early_stop_callback, model_checkpoint],
    accelerator="auto",
    devices=1,
    gradient_clip_val=GRAD_CLIP,
)

# === TRAIN ===
trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)
